# Primary Model

## Problem Definition

**Question.** Which direction classifier best predicts `direction_label` in {-1, 1} without using future event outcomes?

**Role in the workflow.** Select and tune the primary direction model, generate development OOF predictions, and evaluate the sealed holdout once.

**Inputs.** Weighted events, the split contract, and 53 event-start sentiment/fractional-price/technical features.

**Outputs.** Primary model artifact, candidate/tuning/importance tables, and OOF plus holdout side/probability/confidence predictions in `data/model_artifact/`.

**Why this method.** Weighted negative log loss rewards calibrated class probabilities needed by meta-labeling and bet sizing.

**Assumptions.** Only development selects the model; identifiers and all future outcomes are forbidden features; random state is 42.

**Handoff.** Primary OOF predictions to `meta_model.ipynb`, and fixed holdout predictions to final evaluation.


## Test Set Isolation and Development Exploration

The upstream candidate split is reused before class balance, comparison, tuning, or importance is computed. All displayed selection evidence below is development-only.


In [1]:
from pathlib import Path
import sys

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score, log_loss, precision_score
from sklearn.ensemble import RandomForestClassifier

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.strategy_modeling.cross_validation import PurgedKFold
from src.strategy_modeling.feature_importance import get_orthogonal_features
from src.strategy_modeling.model_workflow import (
    build_candidate_classifiers,
    candidate_parameter_grids,
    generate_oof_predictions,
    get_primary_feature_columns,
)

RANDOM_STATE = 42
period = "2025-01-01_2025-12-31"
event_path = PROJECT_ROOT / f"data/research_data/events/aapl_news_modeling_weighted_{period}.parquet"
artifact_dir = PROJECT_ROOT / "data/model_artifact"
artifact_dir.mkdir(parents=True, exist_ok=True)

events = pd.read_parquet(event_path).sort_values("event_start", ignore_index=True)
split_manifest = pd.read_parquet(artifact_dir / "split_manifest.parquet")
development_starts = split_manifest.loc[split_manifest["partition"].eq("development"), "event_start"]
holdout_starts = split_manifest.loc[split_manifest["partition"].eq("holdout"), "event_start"]
development = events[events["event_start"].isin(development_starts)].copy()
holdout = events[events["event_start"].isin(holdout_starts)].copy()
feature_columns = get_primary_feature_columns(events)

development = development.set_index("event_start")
holdout = holdout.set_index("event_start")
X_development = development[feature_columns]
y_development = development["direction_label"].astype("int8")
w_development = development["sample_weight"].astype(float)
t1_development = development["event_end"]
cv = PurgedKFold(n_splits=5, t1=t1_development, pct_embargo=0.01)

assert len(development) + len(holdout) == len(events)
assert len(feature_columns) == 53
assert not set(feature_columns).intersection({"event_end", "vertical_barrier", "target_return", "raw_return", "direction_label", "symbol"})
assert development["event_end"].lt(holdout.index.min()).all()


def score_predictions(predictions):
    return {
        "weighted_log_loss": log_loss(
            y_development,
            np.column_stack([1.0 - predictions["probability"], predictions["probability"]]),
            labels=[-1, 1],
            sample_weight=w_development,
        ),
        "f1": f1_score(y_development, predictions["prediction"], pos_label=1, sample_weight=w_development),
        "precision": precision_score(y_development, predictions["prediction"], pos_label=1, sample_weight=w_development, zero_division=0),
    }


candidates = build_candidate_classifiers(random_state=RANDOM_STATE, n_jobs=1)
comparison_rows = []
candidate_oof = {}
for name, estimator in candidates.items():
    predictions = generate_oof_predictions(estimator, X_development, y_development, w_development, cv, positive_label=1)
    candidate_oof[name] = predictions
    comparison_rows.append({"candidate": name, **score_predictions(predictions)})

comparison = pd.DataFrame(comparison_rows).set_index("candidate").sort_values("weighted_log_loss")
selected_name = comparison.index[0]
display(comparison)


/Users/kwonjunhyuk9/Documents/financial-machine-learning/.venv/lib/python3.11/site-packages/sklearn/ensemble/_bagging.py:919: UserWarning: Some inputs do not have OOB scores. This probably means too few estimators were used to compute any reliable oob estimates.
  warn(
/Users/kwonjunhyuk9/Documents/financial-machine-learning/.venv/lib/python3.11/site-packages/sklearn/ensemble/_bagging.py:925: RuntimeWarning: invalid value encountered in divide
  oob_decision_function = predictions / predictions.sum(axis=1)[:, np.newaxis]


/Users/kwonjunhyuk9/Documents/financial-machine-learning/.venv/lib/python3.11/site-packages/sklearn/ensemble/_bagging.py:919: UserWarning: Some inputs do not have OOB scores. This probably means too few estimators were used to compute any reliable oob estimates.
  warn(
/Users/kwonjunhyuk9/Documents/financial-machine-learning/.venv/lib/python3.11/site-packages/sklearn/ensemble/_bagging.py:925: RuntimeWarning: invalid value encountered in divide
  oob_decision_function = predictions / predictions.sum(axis=1)[:, np.newaxis]


/Users/kwonjunhyuk9/Documents/financial-machine-learning/.venv/lib/python3.11/site-packages/sklearn/ensemble/_bagging.py:919: UserWarning: Some inputs do not have OOB scores. This probably means too few estimators were used to compute any reliable oob estimates.
  warn(
/Users/kwonjunhyuk9/Documents/financial-machine-learning/.venv/lib/python3.11/site-packages/sklearn/ensemble/_bagging.py:925: RuntimeWarning: invalid value encountered in divide
  oob_decision_function = predictions / predictions.sum(axis=1)[:, np.newaxis]


/Users/kwonjunhyuk9/Documents/financial-machine-learning/.venv/lib/python3.11/site-packages/sklearn/ensemble/_bagging.py:919: UserWarning: Some inputs do not have OOB scores. This probably means too few estimators were used to compute any reliable oob estimates.
  warn(
/Users/kwonjunhyuk9/Documents/financial-machine-learning/.venv/lib/python3.11/site-packages/sklearn/ensemble/_bagging.py:925: RuntimeWarning: invalid value encountered in divide
  oob_decision_function = predictions / predictions.sum(axis=1)[:, np.newaxis]


/Users/kwonjunhyuk9/Documents/financial-machine-learning/.venv/lib/python3.11/site-packages/sklearn/ensemble/_bagging.py:919: UserWarning: Some inputs do not have OOB scores. This probably means too few estimators were used to compute any reliable oob estimates.
  warn(
/Users/kwonjunhyuk9/Documents/financial-machine-learning/.venv/lib/python3.11/site-packages/sklearn/ensemble/_bagging.py:925: RuntimeWarning: invalid value encountered in divide
  oob_decision_function = predictions / predictions.sum(axis=1)[:, np.newaxis]


,weighted_log_loss,f1,precision
candidate,,,
logistic_regression,0.670893,0.609936,0.572947
random_forest,0.678850,0.527954,0.540072
boosting,0.690358,0.568178,0.509350
bagging,0.712885,0.536316,0.535942


## Purged Tuning and Development OOF Output

Only the winning family is tuned on the same five purged folds. The resulting OOF predictions are produced by estimators that did not fit their prediction rows; their provenance is stored explicitly for meta-label validation.


In [2]:
tuning_rows = []
tuned_oof_by_configuration = {}
for configuration in candidate_parameter_grids()[selected_name]:
    estimator = clone(candidates[selected_name]).set_params(**configuration)
    predictions = generate_oof_predictions(estimator, X_development, y_development, w_development, cv, positive_label=1)
    key = repr(configuration)
    tuned_oof_by_configuration[key] = predictions
    tuning_rows.append({"configuration": key, **configuration, **score_predictions(predictions)})

tuning = pd.DataFrame(tuning_rows).sort_values("weighted_log_loss", ignore_index=True)
best_configuration = {key: tuning.loc[0, key] for key in candidate_parameter_grids()[selected_name][0]}
tuned_estimator = clone(candidates[selected_name]).set_params(**best_configuration)
primary_oof = tuned_oof_by_configuration[tuning.loc[0, "configuration"]]

primary_oof_output = development[["event_end", "raw_return", "direction_label", "sample_weight"]].copy()
primary_oof_output["partition"] = "development"
primary_oof_output["primary_side"] = primary_oof["prediction"].astype("int8")
primary_oof_output["primary_probability"] = primary_oof["probability"]
primary_oof_output["primary_probability_negative"] = 1.0 - primary_oof["probability"]
primary_oof_output["primary_probability_positive"] = primary_oof["probability"]
primary_oof_output["primary_class_probability"] = np.where(primary_oof_output["primary_side"].eq(1), primary_oof_output["primary_probability_positive"], primary_oof_output["primary_probability_negative"])
primary_oof_output["primary_confidence"] = np.maximum(primary_oof["probability"], 1.0 - primary_oof["probability"])
assert np.allclose(primary_oof_output["primary_class_probability"], primary_oof_output["primary_confidence"])
primary_oof_output["prediction_source"] = primary_oof["prediction_source"]
primary_oof_output["cv_fold"] = primary_oof["fold"]

display(tuning)
display(primary_oof_output.head())


,configuration,model__C,weighted_log_loss,f1,precision
0,{'model__C': 0.1},0.1,0.661683,0.591813,0.570596
1,{'model__C': 1.0},1.0,0.670893,0.609936,0.572947
2,{'model__C': 10.0},10.0,0.697490,0.596163,0.550460


,event_end,raw_return,direction_label,sample_weight,partition,primary_side,primary_probability,primary_probability_negative,primary_probability_positive,primary_class_probability,primary_confidence,prediction_source,cv_fold
event_start,,,,,,,,,,,,,
2025-01-02 15:00:32.232433+00:00,2025-01-02 15:24:06.857548+00:00,-0.005267,-1,0.897596,development,1,0.504461,0.495539,0.504461,0.504461,0.504461,oof,0
2025-01-02 15:32:28.839475+00:00,2025-01-02 15:49:31.568019+00:00,-0.000061,-1,0.191575,development,1,0.702947,0.297053,0.702947,0.702947,0.702947,oof,0
2025-01-02 16:48:23.973430+00:00,2025-01-02 17:24:27.805471+00:00,-0.003145,-1,0.618261,development,-1,0.314649,0.685351,0.314649,0.685351,0.685351,oof,0
2025-01-02 17:35:50.819642+00:00,2025-01-02 18:07:35.869987+00:00,-0.000124,-1,0.007590,development,-1,0.423191,0.576809,0.423191,0.576809,0.576809,oof,0
2025-01-03 15:00:13.717905+00:00,2025-01-03 15:09:26.860772+00:00,0.003910,1,0.565107,development,1,0.628660,0.371340,0.628660,0.628660,0.628660,oof,0


## Development Feature Importance and Error Analysis

MDI, MDA, and SFI use development only. A separate random-forest diagnostic supplies comparable named-feature importances even if another family wins; orthogonal analysis reports redundancy without replacing the final feature schema.


In [3]:
np.random.seed(RANDOM_STATE)
diagnostic_forest = RandomForestClassifier(
    n_estimators=120,
    class_weight="balanced_subsample",
    max_features="sqrt",
    n_jobs=1,
    random_state=RANDOM_STATE,
).fit(X_development, y_development, sample_weight=w_development)

mdi = pd.Series(diagnostic_forest.feature_importances_, index=feature_columns, name="mdi")
mda_result = permutation_importance(
    diagnostic_forest,
    X_development,
    y_development,
    scoring="neg_log_loss",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=1,
    sample_weight=w_development,
)
mda = pd.Series(mda_result.importances_mean, index=feature_columns, name="mda")

sfi_scores = {}
for feature in feature_columns:
    single_predictions = generate_oof_predictions(
        candidates["logistic_regression"],
        X_development[[feature]],
        y_development,
        w_development,
        cv,
        positive_label=1,
    )
    sfi_scores[feature] = -score_predictions(single_predictions)["weighted_log_loss"]
sfi = pd.Series(sfi_scores, name="sfi_neg_log_loss")

importance = pd.concat([mdi, mda, sfi], axis=1).sort_values("mda", ascending=False)
orthogonal = get_orthogonal_features(X_development, var_thres=0.95)
orthogonal_summary = pd.DataFrame(
    {
        "component": orthogonal.columns,
        "label_correlation": [orthogonal[column].corr(y_development) for column in orthogonal.columns],
    }
)

importance.to_parquet(artifact_dir / "primary_feature_importance.parquet")
orthogonal_summary.to_parquet(artifact_dir / "primary_orthogonal_features.parquet", index=False)
display(importance.head(10))
display(orthogonal_summary.head())


,mdi,mda,sfi_neg_log_loss
Force Index,0.049260,0.075597,-0.680678
Balance of Power,0.035301,0.071643,-0.694826
Williams %R,0.024502,0.052242,-0.695170
Relative Vigor Index,0.036886,0.047218,-0.683840
Relative Strength Index,0.035542,0.044744,-0.697537
Stochastic %K,0.022912,0.043321,-0.695170
Stochastic %D,0.029760,0.042456,-0.695917
McClellan Oscillator,0.038014,0.040863,-0.694411
mean_sentiment_score,0.033311,0.040536,-0.691305
Ultimate Oscillator,0.029093,0.040033,-0.698552


,component,label_correlation
0,PC_1,NaN


## Final Fit and One-Time Holdout Evaluation

The tuned family is fitted on all development rows, then the holdout is predicted once. No code below changes the selected family, features, or hyperparameters after seeing holdout metrics.


In [4]:
final_primary = clone(tuned_estimator).fit(
    X_development,
    y_development,
    sample_weight=w_development.to_numpy(),
)
holdout_probability = final_primary.predict_proba(holdout[feature_columns])[:, list(final_primary.classes_).index(1)]
holdout_side = final_primary.predict(holdout[feature_columns]).astype("int8")

primary_holdout_output = holdout[["event_end", "raw_return", "direction_label", "sample_weight"]].copy()
primary_holdout_output["partition"] = "holdout"
primary_holdout_output["primary_side"] = holdout_side
primary_holdout_output["primary_probability"] = holdout_probability
primary_holdout_output["primary_probability_negative"] = 1.0 - holdout_probability
primary_holdout_output["primary_probability_positive"] = holdout_probability
primary_holdout_output["primary_class_probability"] = np.where(primary_holdout_output["primary_side"].eq(1), primary_holdout_output["primary_probability_positive"], primary_holdout_output["primary_probability_negative"])
primary_holdout_output["primary_confidence"] = np.maximum(holdout_probability, 1.0 - holdout_probability)
assert np.allclose(primary_holdout_output["primary_class_probability"], primary_holdout_output["primary_confidence"])
primary_holdout_output["prediction_source"] = "holdout"
primary_holdout_output["cv_fold"] = pd.NA

holdout_metrics = pd.Series(
    {
        "weighted_log_loss": log_loss(holdout["direction_label"], np.column_stack([1.0 - holdout_probability, holdout_probability]), labels=[-1, 1], sample_weight=holdout["sample_weight"]),
        "f1": f1_score(holdout["direction_label"], holdout_side, pos_label=1, sample_weight=holdout["sample_weight"]),
        "precision": precision_score(holdout["direction_label"], holdout_side, pos_label=1, sample_weight=holdout["sample_weight"], zero_division=0),
    },
    name="holdout",
)

primary_predictions = pd.concat([primary_oof_output, primary_holdout_output]).sort_index()
primary_predictions.to_parquet(artifact_dir / "primary_predictions.parquet")
comparison.to_parquet(artifact_dir / "primary_candidate_metrics.parquet")
tuning.to_parquet(artifact_dir / "primary_tuning_metrics.parquet", index=False)
holdout_metrics.to_frame().to_parquet(artifact_dir / "primary_holdout_metrics.parquet")
joblib.dump(
    {
        "estimator": final_primary,
        "feature_columns": feature_columns,
        "selected_candidate": selected_name,
        "best_configuration": best_configuration,
        "holdout_boundary": holdout.index.min(),
        "random_state": RANDOM_STATE,
    },
    artifact_dir / "primary_model.joblib",
)

display(holdout_metrics.to_frame())
print(artifact_dir / "primary_model.joblib")


,holdout
weighted_log_loss,0.711875
f1,0.577434
precision,0.579253


/Users/kwonjunhyuk9/Documents/financial-machine-learning/data/model_artifact/primary_model.joblib


## Results, Limitations, and Handoff

Development OOF performance includes model-family and grid-selection uncertainty, and the one-year AAPL holdout is small. The stored holdout result is final even if it is worse than development; no retuning follows.

The next notebook receives OOF-only development sides/confidences and fixed holdout predictions. No conclusion in this notebook is evidence of live-trading profitability.
